# Week 1 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-01.md`](../../weeks/week-01.md) · **Given engine:** [`eliza.py`](eliza.py)

This notebook is the **teaching walkthrough** of the week-1 studio. It **imports the
reference code from [`solution.py`](solution.py)** — it never re-pastes it — so the
answer shown here is byte-for-byte the one the provided test grades. The correctness
guarantee is separate: `python3 studios/_verify_solutions.py week-01` runs the
unmodified `test_eliza.py` against `solution.py`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [1]:
# --- bootstrap: put the week folder (for solution/eliza) and the repo root
# (for aicourse) on the path, so this runs from anywhere. ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-01"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect, random
import eliza
import solution

## Task 1a — three new rules

The bar: **≥ 3** rules, each a *real* trigger (no pattern that matches the empty
string, or it shadows the fallback for everything), and **none** duplicating a base
pattern. Here is the reference set, straight from `solution.py`:

In [2]:
print(inspect.getsource(solution.new_rules))
base = {p for p, _ in eliza.RULES}
for pat, templates in solution.new_rules():
    import re
    assert re.search(pat, "") is None, "would fire on everything"
    assert pat not in base, "duplicates a base rule"
    print("novel & real-trigger:", pat)

def new_rules():
    """Three novel rules. Each has a real trigger (none matches the empty
    string) and none duplicates a base pattern in ``eliza.RULES``."""
    return [
        (r"\bI want (.*)", ["What would it mean to you to get {0}?",
                            "Why do you want {0}?"]),
        (r"\bI can't (.*)", ["What makes you feel you can't {0}?",
                             "Have you ever been able to {0}?"]),
        (r"\bI think (.*)", ["Do you really think {0}?",
                             "What led you to think {0}?"]),
    ]

novel & real-trigger: \bI want (.*)
novel & real-trigger: \bI can't (.*)
novel & real-trigger: \bI think (.*)


## Task 1b — the extended responder

Base rules **win** (so nothing added regresses the demo), then the new rules, then
the fallback. Same loop as `eliza.respond`, over the joined table — we *reuse* the
engine's `reflect`, we do not reinvent it.

In [3]:
print(inspect.getsource(solution.extended_respond))
random.seed(1)
# base rule still wins — the currency of Task 1b:
reply = solution.extended_respond("I feel anxious most of the time")
print("base-rule reply :", reply)
assert reply.lower().startswith("do you often feel") and "anxious" in reply.lower()
# a new rule fires when no base rule matches:
random.seed(1)
print("new-rule reply  :", solution.extended_respond("I want to pass this course"))

def extended_respond(text):
    """Base rules FIRST (so the demo never regresses), then ``new_rules()``,
    then the fallback. Same loop as ``eliza.respond``, over the joined table."""
    for pattern, templates in RULES + new_rules():
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            groups = [reflect(g) for g in m.groups()]
            return random.choice(templates).format(*groups)
    return random.choice(FALLBACK)

base-rule reply : Do you often feel anxious most of the time?
new-rule reply  : What would it mean to you to get to pass this course?


## Task 1c — the deliberate failure (watch the guarantee break)

This is the payload of week 1. The declared failure is a **compound sentence** whose
*second clause is dropped entirely* — first-match-wins means ELIZA never inspects it.
Below we reproduce exactly what `test_declared_failure_is_hollow` checks: none of
clause B's content words survive into the reply.

In [4]:
import re
text, why = solution.failure_case()
random.seed(1)
reply = solution.extended_respond(text)
print("input :", text)
print("reply :", reply)
print("why   :", why)

_, clause_b = text.lower().split(" and ", 1)
content = [w for w in re.findall(r"[a-z]+", clause_b) if len(w) >= 4]
surfaced = [w for w in content if w in reply.lower()]
print("\nclause B content words:", content)
print("surfaced in the reply :", surfaced, "  <-- empty = the guarantee failed, as designed")
assert surfaced == [], "clause B leaked; this would not be an ELIZA-class drop"
print("\nGUARANTEE FAIL confirmed: ELIZA answered fluently while ignoring half the sentence.")

input : My mother is a doctor and my father is a lawyer
reply : Tell me more about your mother.
why   : First-match-wins: the 'my (mother|father|family)' rule fires on clause A, so clause B ('my father is a lawyer') is never inspected — fluent, on-topic, and half-blind.

clause B content words: ['father', 'lawyer']
surfaced in the reply : []   <-- empty = the guarantee failed, as designed

GUARANTEE FAIL confirmed: ELIZA answered fluently while ignoring half the sentence.


## Task 2 — first contact (ELIZA vs. a real model)

Students send the **same** prompts to ELIZA and to a local model, record both
transcripts, and answer the README's three questions. Here we run the comparison
headless with the deterministic `echo` backend so the notebook executes anywhere;
**for a real comparison, re-run with `backend="ollama"` or `backend="manual"`** — the
`echo` backend is a fake and its text must never be reported as a model result.

The anti-pattern to name for students: **do not ask the model to grade itself.** You
are the checker (scorecard axis 3).

In [5]:
import os, json
from aicourse.llm import LLM

prompts = json.load(open(week / "prompts.json"))["first_contact"]
backend = os.environ.get("AICOURSE_NB_BACKEND", "echo")   # echo = headless-safe fake
llm = LLM(backend=backend)

random.seed(1)
for p in prompts[:4]:
    print("you  >", p)
    print("ELIZA>", solution.extended_respond(p))
    print("LLM  >", llm.complete(p).text)
    print()

you  > I am not feeling great about the exam
ELIZA> How long have you been not feeling great about the exam?
LLM  > [echo backend] 37 chars received. This is NOT a model response.

you  > I feel anxious most of the time
ELIZA> Do you often feel anxious most of the time?
LLM  > [echo backend] 31 chars received. This is NOT a model response.

you  > My mother thinks I should study more
ELIZA> Tell me more about your mother.
LLM  > [echo backend] 36 chars received. This is NOT a model response.

you  > I did badly because I did not sleep
ELIZA> Is that the real reason?
LLM  > [echo backend] 35 chars received. This is NOT a model response.



### What the students must write

1. Where did ELIZA feel *more* honest than the model (named its limits vs. bluffed)?
2. The model has no more *understanding* than ELIZA — what does it have *more* of, and
   why does that make its failures **harder** to predict?
3. Name one input for which you can **prove** what ELIZA will do, and one for which you
   cannot prove what the model will do. That gap is the whole course.

They read their week-1 answer again in week 14.